In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Verifica las versiones instaladas en tu sesión de Colab
print(f'Matplotlib: {plt.matplotlib.__version__}')   # anota la versión
print(f'Seaborn:    {sns.__version__}')

# --- Configuración global de estilo ---
sns.set_theme(style='whitegrid')   # fondo blanco con cuadrícula suave
plt.rcParams['figure.dpi'] = 120    # mayor resolución en Colab

In [ ]:
# Cargar Titanic (mismo dataset de la sección de pandas)
titanic = sns.load_dataset('titanic')

# Revisar NaN en age antes de imputar
n_nan_age = titanic['age'].isnull().sum()
print(f'Valores faltantes en age: {n_nan_age}')  # 177

# Imputar con la mediana para que las gráficas no omitan filas
titanic['age'] = titanic['age'].fillna(titanic['age'].median())
print(f'NaN restantes en age: {titanic["age"].isnull().sum()}')  # 0

print(f'Filas: {titanic.shape[0]}  |  Columnas: {titanic.shape[1]}')

In [ ]:
# Histograma de edad con curva de densidad superpuesta
fig, ax = plt.subplots(figsize=(8, 4))

sns.histplot(
    data=titanic,
    x='age',
    bins=30,
    kde=True,           # curva de densidad kernel
    color='steelblue',
    ax=ax
)

ax.set_title('Distribución de edad — Pasajeros del Titanic')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Número de pasajeros')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: distribución de edad según si sobrevivió o no
fig, ax = plt.subplots(figsize=(7, 4))

sns.boxplot(
    data=titanic,
    x='survived',
    y='age',
    hue='survived', # Asignar 'survived' a hue para el color
    palette={0: '#E57373', 1: '#64B5F6'},   # Usar claves enteras para la paleta
    legend=False, # Suprimir la leyenda redundante
    ax=ax
)

ax.set_title('Distribución de edad según supervivencia')
ax.set_xlabel('Sobrevivió  (0 = No,  1 = Sí)')
ax.set_ylabel('Edad (años)')

plt.tight_layout()
plt.show()

In [ ]:
# Countplot: sobrevivientes por clase de pasajero
fig, ax = plt.subplots(figsize=(7, 4))

sns.countplot(
    data=titanic,
    x='pclass',
    hue='survived',
    palette={0: '#E57373', 1: '#64B5F6'},
    ax=ax
)

ax.set_title('Supervivencia por clase de pasajero')
ax.set_xlabel('Clase (1 = Primera,  2 = Segunda,  3 = Tercera)')
ax.set_ylabel('Número de pasajeros')
ax.legend(title='Sobrevivió', labels=['No', 'Sí'])

plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap de correlaciones entre features ---
# Excluimos survived: la analizamos por separado al final
features = ['pclass', 'age', 'sibsp', 'parch', 'fare']

matriz_corr = titanic[features].corr()

fig, ax = plt.subplots(figsize=(6, 5))

sns.heatmap(
    matriz_corr,
    annot=True,          # muestra el valor numérico en cada celda
    fmt='.2f',           # dos decimales
    cmap='coolwarm',     # azul=negativa, blanco=cero, rojo=positiva
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax
)

ax.set_title('Correlación entre features — Titanic')
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlación de cada feature con el target ---
# .corr()['survived'] calcula la correlación de Pearson de cada
# feature contra la columna survived
corr_target = (
    titanic[features + ['survived']]
    .corr()['survived']
    .drop('survived')          # excluir la autocorrelación (=1)
    .abs()                     # valor absoluto: el signo de la correlación
                            # con un target 0/1 depende de qué categoría
                            # se codificó como 0 y cuál como 1 — un detalle
                            # de codificación, no de la relación real.
                            # Para diagnóstico de relevancia importa
                            # la magnitud, no la dirección.
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(6, 3))
corr_target.plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Correlación (Pearson) con survived')
ax.set_ylabel('|correlación|')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Panel 2×2 con cuatro gráficas del dataset Titanic
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

# --- [0,0] Histograma de edad ---
sns.histplot(data=titanic, x='age', bins=25, kde=True,
            color='steelblue', ax=axs[0, 0])
axs[0, 0].set_title('Distribución de edad')
axs[0, 0].set_xlabel('Edad')

# --- [0,1] Tasa de supervivencia por sexo ---
sns.barplot(data=titanic, x='sex', y='survived',
            hue='sex', # Asignar 'sex' a hue para el color
            palette=['#64B5F6', '#E57373'],
            legend=False, # Suprimir la leyenda redundante
            ax=axs[0, 1])
axs[0, 1].set_title('Tasa de supervivencia por sexo')
axs[0, 1].set_ylabel('Proporción que sobrevivió')

# --- [1,0] Distribución de tarifa (fare) ---
sns.histplot(data=titanic, x='fare', bins=40,
            color='#66BB6A', ax=axs[1, 0])
axs[1, 0].set_title('Distribución de tarifa')
axs[1, 0].set_xlabel('Tarifa (£)')

# --- [1,1] Edad vs tarifa coloreado por clase ---
sns.scatterplot(data=titanic, x='age', y='fare',
                hue='pclass', palette='Set2',
                alpha=0.6, ax=axs[1, 1])
axs[1, 1].set_title('Edad vs. tarifa por clase')

fig.suptitle('Exploración visual — Titanic', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Guardar la figura del panel 2×2 en Drive
from pathlib import Path

# Asegúrate de haber montado Drive antes (ver Capítulo 1)
ruta_salida = Path('/content/drive/MyDrive/ML_con_sklearn/figuras')

try:
    ruta_salida.mkdir(parents=True, exist_ok=True)
    fig.savefig(ruta_salida / 'eda_titanic.png',
                dpi=150,         # 150 dpi para uso en reportes
                bbox_inches='tight')   # recorta márgenes sobrantes
    print(f'Figura guardada en {ruta_salida / "eda_titanic.png"}')
except Exception as e:
    print(f'Error al guardar: {e}')
    print('Verifica que Drive esté montado (drive.mount)')